In [6]:
"""
=============================================================================
  Spaceship Titanic Passenger Transportation Classification
  Kaggle Competition: Spaceship Titanic
  Author  : Kautsar Hilmi
  Metric  : Classification Accuracy
=============================================================================
"""

# ─── 0. IMPORTS ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import xgboost as xgb
import lightgbm as lgb

# ─── 1. LOAD DATA ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  STEP 1: LOADING DATA")
print("=" * 60)

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(f"  Train shape : {train.shape}")
print(f"  Test  shape : {test.shape}")

test_id = test["PassengerId"]
y_train = train["Transported"].astype(int) # Convert boolean to 0/1

# ─── 2. COMBINE DATA FOR PROCESSING ──────────────────────────────────────────
train_raw = train.drop(["PassengerId", "Transported"], axis=1)
test_raw = test.drop(["PassengerId"], axis=1)
ntrain = train_raw.shape[0]

all_data = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)

# ─── 3. ADVANCED FEATURE ENGINEERING ─────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 2: FEATURE ENGINEERING & PARSING")
print("=" * 60)

# 3a. Extract Group and GroupSize from PassengerId
# PassengerId format: gggg_pp (gggg = Group, pp = Passenger number within group)
passenger_ids = pd.concat([train["PassengerId"], test["PassengerId"]], axis=0, ignore_index=True)
all_data["Group"] = passenger_ids.apply(lambda x: x.split("_")[0])
all_data["GroupSize"] = all_data.groupby("Group")["Group"].transform("count")
all_data["IsSolo"] = (all_data["GroupSize"] == 1).astype(int)

# 3b. Split Cabin (deck/num/side)
all_data["Cabin"] = all_data["Cabin"].fillna("Missing/Missing/Missing")
all_data["Cabin_deck"] = all_data["Cabin"].apply(lambda x: x.split("/")[0])
all_data["Cabin_num"] = all_data["Cabin"].apply(lambda x: x.split("/")[1])
all_data["Cabin_side"] = all_data["Cabin"].apply(lambda x: x.split("/")[2])

# Convert Cabin_num back to numeric, replace 'Missing' placeholder with NaN
all_data["Cabin_num"] = pd.to_numeric(all_data["Cabin_num"], errors='coerce')
all_data["Cabin_num"] = all_data["Cabin_num"].fillna(all_data["Cabin_num"].median())

all_data.drop(["Cabin", "Group"], axis=1, inplace=True)

# 3c. Expenditure Aggregation
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
for col in spend_cols:
    all_data[col] = all_data[col].fillna(0)

all_data["TotalSpending"] = all_data[spend_cols].sum(axis=1)
all_data["IsSpender"] = (all_data["TotalSpending"] > 0).astype(int)

# Log transform spending to handle extreme right skewness
all_data["TotalSpending"] = np.log1p(all_data["TotalSpending"])

# 3d. Age Binning
all_data["Age"] = all_data.groupby("HomePlanet")["Age"].transform(lambda x: x.fillna(x.median()))
all_data["AgeGroup"] = pd.cut(all_data["Age"], bins=[-1, 12, 18, 35, 60, 100], labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"])

print("  Features successfully engineered: GroupSize, IsSolo, Cabin_deck, Cabin_side, TotalSpending, IsSpender, AgeGroup")

# ─── 4. CONTEXTUAL IMPUTATION & ENCODING ─────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 3: CONTEXTUAL IMPUTATION & ONE-HOT ENCODING")
print("=" * 60)

# Impute remaining categorical values with mode
cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP", "Cabin_deck", "Cabin_side", "AgeGroup"]
for col in cat_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# Drop structural non-predictive columns
all_data.drop(["Name"], axis=1, errors='ignore', inplace=True)

# One-Hot Encoding for categorical features
all_data = pd.get_dummies(all_data, columns=cat_cols, drop_first=True)
print(f"  Dataset shape after encoding: {all_data.shape}")

# Split back to Train / Test
X_train = all_data[:ntrain]
X_test = all_data[ntrain:]

# ─── 5. ENSEMBLE MODEL DEFINITION (STACKING) ─────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 4: MODEL DEFINITIONS & STACKING ARCHITECTURE")
print("=" * 60)

# Base Learners
rf_model = RandomForestClassifier(n_estimators=500, max_depth=10, random_state=42, n_jobs=-1)
xgb_model = xgb.XGBRegressor(n_estimators=1000, max_depth=5, learning_rate=0.03, random_state=42, n_jobs=-1, eval_metric="logloss")
lgb_model = lgb.LGBMClassifier(n_estimators=1000, max_depth=5, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)

# Meta-Learner (Logistic Regression Stack)
estimators = [
    ('rf', rf_model),
    ('xgb', xgb_model),
    ('lgb', lgb_model)
]
stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

# ─── 6. STRATIFIED K-FOLD CROSS-VALIDATION ───────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 5: STRATIFIED K-FOLD CROSS-VALIDATION (k=5)")
print("=" * 60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(stacking_model, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1)

for fold, score in enumerate(cv_scores):
    print(f"  Fold {fold+1} Accuracy: {score:.5f}")
print(f"  Mean Cross-Validation Accuracy: {cv_scores.mean():.5f} ± {cv_scores.std():.5f}")

# ─── 7. FINAL TRAINING & SUBMISSION GENERATION ───────────────────────────────
print("\n" + "=" * 60)
print("  STEP 6: FINAL TRAINING & SUBMISSION GENERATION")
print("=" * 60)

stacking_model.fit(X_train, y_train)
final_preds = stacking_model.predict(X_test)

# Convert 0/1 predictions back to True/False strings/booleans as requested by Kaggle
final_preds_bool = final_preds.astype(bool)

submission = pd.DataFrame({
    "PassengerId": test_id,
    "Transported": final_preds_bool
})

submission.to_csv("submission.csv", index=False)
print("  ✅ submission.csv saved successfully!")
print("=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)

  STEP 1: LOADING DATA
  Train shape : (8693, 14)
  Test  shape : (4277, 13)

  STEP 2: FEATURE ENGINEERING & PARSING
  Features successfully engineered: GroupSize, IsSolo, Cabin_deck, Cabin_side, TotalSpending, IsSpender, AgeGroup

  STEP 3: CONTEXTUAL IMPUTATION & ONE-HOT ENCODING
  Dataset shape after encoding: (12970, 31)

  STEP 4: MODEL DEFINITIONS & STACKING ARCHITECTURE

  STEP 5: STRATIFIED K-FOLD CROSS-VALIDATION (k=5)
  Fold 1 Accuracy: 0.82059
  Fold 2 Accuracy: 0.79988
  Fold 3 Accuracy: 0.81024
  Fold 4 Accuracy: 0.81703
  Fold 5 Accuracy: 0.79517
  Mean Cross-Validation Accuracy: 0.80858 ± 0.00973

  STEP 6: FINAL TRAINING & SUBMISSION GENERATION
  ✅ submission.csv saved successfully!
  PIPELINE COMPLETE
